# A little extra!

## New addition to Week 1

### The Unreasonable Effectiveness of the Agent Loop

# What is an Agent?

## Three competing definitions

1. AI systems that can do work for you independently - Sam Altman

2. A system in which an LLM controls the workflow - Anthropic

3. An LLM agent runs tools in a loop to achieve a goal

## The third one is the new, emerging definition

But what does it mean?

Let's make it real.

In [5]:
# Start with some imports - rich is a library for making formatted text output in the terminal

from rich.console import Console
from dotenv import load_dotenv
from openai import OpenAI
import json
import os
load_dotenv(override=True)

True

In [6]:
def show(text):
    try:
        Console().print(text)
    except Exception:
        print(text)

In [7]:
url = os.getenv("URL")
api_key = os.getenv("NVIDIA_API_KEY")
openai = OpenAI(
    api_key=api_key,
    base_url=url
)

In [8]:
# Some lists!

todos = []
completed = []

In [9]:
def get_todo_report() -> str:
    result = ""
    for index, todo in enumerate(todos):
        if completed[index]:
            result += f"Todo #{index + 1}: [green][strike]{todo}[/strike][/green]\n"
        else:
            result += f"Todo #{index + 1}: {todo}\n"
    show(result)
    return result

In [10]:
get_todo_report()

''

In [11]:
def create_todos(descriptions: list[str]) -> str:
    todos.extend(descriptions)
    completed.extend([False] * len(descriptions))
    return get_todo_report()

In [12]:
def mark_complete(index: int, completion_notes: str) -> str:
    if 1 <= index <= len(todos):
        completed[index - 1] = True
    else:
        return "No todo at this index."
    Console().print(completion_notes)
    return get_todo_report()

In [13]:
todos, completed = [], []

create_todos(["Buy groceries", "Finish extra lab", "Eat banana"])

Todo #1: Buy groceries
Todo #2: Finish extra lab
Todo #3: Eat banana

'Todo #1: Buy groceries\nTodo #2: Finish extra lab\nTodo #3: Eat banana\n'

In [14]:
mark_complete(1, "bought")

bought

Todo #1: Buy groceries
Todo #2: Finish extra lab
Todo #3: Eat banana

'Todo #1: [green][strike]Buy groceries[/strike][/green]\nTodo #2: Finish extra lab\nTodo #3: Eat banana\n'

In [15]:
create_todos_json = {
    "name": "create_todos",
    "description": "Add new todos from a list of descriptions and return the full list",
    "parameters": {
        "type": "object",
        "properties": {
            "descriptions": {
                'type': 'array',
                'items': {'type': 'string'},
                'title': 'Descriptions'
                }
            },
        "required": ["descriptions"],
        "additionalProperties": False
    }
}

In [16]:
mark_complete_json = {
    "name": "mark_complete",
    "description": "Mark complete the todo at the given position (starting from 1) and return the full list",
    "parameters": {
        'properties': {
            'index': {
                'description': 'The 1-based index of the todo to mark as complete',
                'title': 'Index',
                'type': 'integer'
                },
            'completion_notes': {
                'description': 'Notes about how you completed the todo in rich console markup',
                'title': 'Completion Notes',
                'type': 'string'
                }
            },
        'required': ['index', 'completion_notes'],
        'type': 'object',
        'additionalProperties': False
    }
}

In [17]:
tools = [{"type": "function", "function": create_todos_json},
        {"type": "function", "function": mark_complete_json}]

In [23]:
def handle_tool_calls(tool_calls):
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        tool = globals().get(tool_name)
        result = tool(**arguments) if tool else {}
        results.append({"role": "tool","content": json.dumps(result),"tool_call_id": tool_call.id})
    return results

In [27]:
def loop(messages):
    done = False
    while not done:
        
        
        response = openai.chat.completions.create(model="deepseek-ai/deepseek-v3.2", messages=messages, tools=tools, extra_body={"chat_template_kwargs": {"thinking":False}},)


        finish_reason = response.choices[0].finish_reason
        if finish_reason=="tool_calls":
            message = response.choices[0].message
            tool_calls = message.tool_calls
            results = handle_tool_calls(tool_calls)
            messages.append(message)
            messages.extend(results)
        else:
            done = True
    show(response.choices[0].message.content)

In [28]:
system_message = """
You are given a problem to solve, by using your todo tools to plan a list of steps, then carrying out each step in turn.
Now use the todo list tools, create a plan, carry out the steps, and reply with the solution.
If any quantity isn't provided in the question, then include a step to come up with a reasonable estimate.
Provide your solution in Rich console markup without code blocks.
Do not ask the user questions or clarification; respond only with the answer after using your tools.
"""
user_message = """"
A train leaves Boston at 2:00 pm traveling 60 mph.
Another train leaves New York at 3:00 pm traveling 80 mph toward Boston.
When do they meet?
"""
messages = [{"role": "system", "content": system_message}, {"role": "user", "content": user_message}]

In [29]:
todos, completed = [], []
loop(messages)

Todo #1: Determine the distance between Boston and New York
Todo #2: Calculate the position of the first train when the second train starts
Todo #3: Set up the equation for when they meet
Todo #4: Solve for meeting time
Todo #5: Convert solution to readable time format

The distance between Boston and New York is approximately **210 miles**. This is a common estimate for the rail 
distance between the two cities, though actual distances can vary between 210-215 miles depending on the specific 
route.

Todo #1: Determine the distance between Boston and New York
Todo #2: Calculate the position of the first train when the second train starts
Todo #3: Set up the equation for when they meet
Todo #4: Solve for meeting time
Todo #5: Convert solution to readable time format

The first train leaves Boston at 2:00 pm traveling 60 mph. By 3:00 pm (when the second train starts), the first 
train has been traveling for **1 hour**.  
Distance traveled by first train = 60 mph × 1 hour = **60 miles** from Boston toward New York.

Todo #1: Determine the distance between Boston and New York
Todo #2: Calculate the position of the first train when the second train starts
Todo #3: Set up the equation for when they meet
Todo #4: Solve for meeting time
Todo #5: Convert solution to readable time format

...

Solving the equation:

**60t + 80t = 150**  
**140t = 150**  
**t = 150 ÷ 140**  
**t = 15/14 hours**  
**t ≈ 1.0714 hours**

Convert to minutes: 0.0714 × 60 = **≈ 4.29 minutes**

So they meet approximately **1 hour and 4.3 minutes** after 3:00 pm.

Todo #1: Determine the distance between Boston and New York
Todo #2: Calculate the position of the first train when the second train starts
Todo #3: Set up the equation for when they meet
Todo #4: Solve for meeting time
Todo #5: Convert solution to readable time format

The trains meet **1 hour and 4.3 minutes after 3:00 pm**.
- 3:00 pm + 1 hour = **4:00 pm**
- 4:00 pm + 4.3 minutes = **4:04:18 pm** (4 minutes and 18 seconds past 4:00 pm)

**Final answer:** The trains meet at approximately **4:04 pm**.

Todo #1: Determine the distance between Boston and New York
Todo #2: Calculate the position of the first train when the second train starts
Todo #3: Set up the equation for when they meet
Todo #4: Solve for meeting time
Todo #5: Convert solution to readable time format

The two trains meet at approximately **4:04 pm**.

**Solution Summary:**
1. Distance between Boston and New York: ~210 miles
2. First train travels from 2:00-3:00 pm: 60 miles
3. Remaining distance at 3:00 pm: 150 miles
4. Combined speed after 3:00 pm: 140 mph (60 + 80)
5. Time to meet after 3:00 pm: 150/140 = 15/14 ≈ 1.0714 hours = 1 hour 4.3 minutes
6. Meeting time: 3:00 pm + 1h 4.3m = **4:04 pm**

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercise</h2>
            <span style="color:#ff7800;">Now try to build an Agent Loop from scratch yourself!<br/>
            Create a new .ipynb and make one from first principles, referring back to this as needed.<br/>
            It's one of the few times that I recommend typing from scratch - it's a very satisfying result.
            </span>
        </td>
    </tr>
</table>